# Bronze Layer: Ingestion & Enrichment

### 1. Define schemas

In [0]:
from pyspark.sql.types import *

ratings_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("rating",       FloatType(),    True),
    StructField("timestamp",    LongType(),     True),
])

movies_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("title",    StringType(),   True),
    StructField("genres",   StringType(),   True),
])

links_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("imdbId",   IntegerType(),  True),
    StructField("tmdbId",   IntegerType(),  True),
])

tags_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("tag",          StringType(),   True),
    StructField("timestamp",    LongType(),     True),
])

enrichment_schema = StructType([
    StructField("movieId",           IntegerType()),
    StructField("tmdbId",            IntegerType()),
    StructField("title",             StringType()),
    StructField("directors",         ArrayType(StringType())),
    StructField("budget",            LongType()),
    StructField("revenue",           LongType()),
    StructField("runtime",           IntegerType()),
    StructField("release_date",      StringType()),
    StructField("poster_url",        StringType()),
    StructField("overview",          StringType()),
    StructField("vote_average",      FloatType()),
    StructField("original_language", StringType()),
])

### 2. Read each CSV

In [0]:
df_ratings = spark.read.csv("/Volumes/workspace/default/movie_recommender/ratings.csv", header=True, schema=ratings_schema)
df_movies  = spark.read.csv("/Volumes/workspace/default/movie_recommender/movies.csv",  header=True, schema=movies_schema)
df_links   = spark.read.csv("/Volumes/workspace/default/movie_recommender/links.csv",   header=True, schema=links_schema)
df_tags    = spark.read.csv("/Volumes/workspace/default/movie_recommender/tags.csv",    header=True, schema=tags_schema)

### 3. Add ingestion metadata

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def add_metadata(df, source_name):
    return df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_file", lit(source_name))

### 4. Write to bronze/ as parquet

In [0]:
add_metadata(df_ratings, "ratings.csv") \
    .write.mode("overwrite").parquet("/Volumes/workspace/default/movie_recommender/bronze/ratings")

add_metadata(df_movies, "movies.csv") \
    .write.mode("overwrite").parquet("/Volumes/workspace/default/movie_recommender/bronze/movies")

add_metadata(df_links, "links.csv") \
    .write.mode("overwrite").parquet("/Volumes/workspace/default/movie_recommender/bronze/links")

add_metadata(df_tags, "tags.csv") \
    .write.mode("overwrite").parquet("/Volumes/workspace/default/movie_recommender/bronze/tags")

### 5. Sanity checks

In [0]:
for name, df in [("ratings", df_ratings), ("movies", df_movies),
                  ("links", df_links), ("tags", df_tags)]:
    print(f"\n=== {name} ===")
    print(f"Rows: {df.count()}")
    df.printSchema()
    df.show(3)

---

## TMDB Enrichment

### 6. Load enrichment JSON to parquet

In [0]:
import json

with open("/Volumes/workspace/default/movie_recommender/scraped_metadata.json") as f:
    records = json.load(f)

df_enrichment = (
    spark.createDataFrame(records, enrichment_schema)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("scraped_metadata.json"))
)

df_enrichment.write.mode("overwrite").parquet("/Volumes/workspace/default/movie_recommender/bronze/enrichment")

print(f"Done. {df_enrichment.count()} rows written to /Volumes/workspace/default/movie_recommender/bronze/enrichment")
df_enrichment.show(5, truncate=False)